In [ ]:
# # 패키지 설치 (Julia REPL에서)
# using Pkg
# Pkg.add(["ArchGDAL", "Shapefile", "Plots", "PlotlyJS", "GeoInterface", "GeoJSON"])
# # 하나만 설치
# using Pkg
# Pkg.add("ColorSchemes")

In [ ]:
pwd()

## 지도 그리기
값은 없고 형태만 그림 //
폴리곤이 하나만 읽힌다.

In [ ]:
using ArchGDAL
using Plots

In [ ]:
file_path = "/Users/mkim/Library/Mobile Documents/com~apple~CloudDocs/2025 연구/CA2025/Korea_map/Korea_map.shp"

In [ ]:
# shp 파일 읽기
ArchGDAL.read(file_path) do dataset
    p = plot(size=(1000, 800))
    
    for i in 0:(ArchGDAL.nlayer(dataset)-1)
        layer = ArchGDAL.getlayer(dataset, i)
        
        for feature in layer
            geom = ArchGDAL.getgeom(feature, 0)
            
            if ArchGDAL.getgeomtype(geom) == ArchGDAL.wkbPolygon
                # 외곽선 좌표 추출
                ring = ArchGDAL.getgeom(geom, 0)
                n_points = ArchGDAL.ngeom(ring)
                
                x = Float64[]
                y = Float64[]
                
                for j in 0:(n_points-1)
                    point = ArchGDAL.getgeom(ring, j)
                    push!(x, ArchGDAL.getx(point, 0))
                    push!(y, ArchGDAL.gety(point, 0))
                end
                
                plot!(x, y,
                      seriestype=:shape,
                      fillalpha=0.5,
                      fillcolor=:lightgreen,
                      linecolor=:black,
                      linewidth=0.8,
                      label="")
            end
        end
    end
    
    plot!(title="ArchGDAL",
          xlabel="Longitude",
          ylabel="latitude",
          aspect_ratio=:equal)
    
    display(p)
end

## 지도 그리기 2
값은 없음 // 모든 폴리곤을 다 그렸다.

In [ ]:
using ArchGDAL
using Plots

In [ ]:
function plot_korea_map(file_path::String; 
                       map_title::String="Korea Map",
                       fill_color=:lightblue,
                       line_color=:navy,
                       line_width=1.2,
                       fill_alpha=0.7,
                       image_size=(1200, 900))
    
    # Initialize plot
    p = plot(size=image_size, 
             background_color=:white,
             margin=5Plots.mm)
    
    try
        ArchGDAL.read(file_path) do dataset
            println("Dataset loaded successfully. Number of layers: $(ArchGDAL.nlayer(dataset))")
            
            polygon_count = 0
            
            for i in 0:(ArchGDAL.nlayer(dataset)-1)
                layer = ArchGDAL.getlayer(dataset, i)
                println("Processing layer $(i+1)...")
                
                for feature in layer
                    geom = ArchGDAL.getgeom(feature, 0)
                    
                    if geom === nothing
                        continue
                    end
                    
                    geom_type = ArchGDAL.getgeomtype(geom)
                    
                    # Handle both Polygon and MultiPolygon
                    if geom_type == ArchGDAL.wkbPolygon
                        plot_polygon!(p, geom, fill_color, line_color, line_width, fill_alpha)
                        polygon_count += 1
                        
                    elseif geom_type == ArchGDAL.wkbMultiPolygon
                        # For MultiPolygon, process each polygon individually
                        n_polygons = ArchGDAL.ngeom(geom)
                        for k in 0:(n_polygons-1)
                            poly = ArchGDAL.getgeom(geom, k)
                            plot_polygon!(p, poly, fill_color, line_color, line_width, fill_alpha)
                            polygon_count += 1
                        end
                    end
                end
            end
            
            println("Total $(polygon_count) polygons plotted.")
        end
        
    catch e
        println("Error occurred: $e")
        println("Please check the file path: $file_path")
        return nothing
    end
    
    # Style the plot
    plot!(title=map_title,
          titlefontsize=16,
          xlabel="Longitude",
          ylabel="Latitude",
          xlabelfontsize=12,
          ylabelfontsize=12,
          aspect_ratio=:equal,
          grid=true,
          gridwidth=1,
          gridcolor=:lightgray,
          gridalpha=0.3,
          legend=false,
          dpi=300)
    
    return p
end

In [ ]:
# Helper function to plot individual polygons
function plot_polygon!(p, geom, fill_color, line_color, line_width, fill_alpha)
    try
        # Process exterior ring
        ring = ArchGDAL.getgeom(geom, 0)
        if ring === nothing
            return
        end
        
        n_points = ArchGDAL.ngeom(ring)
        if n_points < 3  # Polygon needs at least 3 points
            return
        end
        
        x = Float64[]
        y = Float64[]
        
        for j in 0:(n_points-1)
            point = ArchGDAL.getgeom(ring, j)
            if point !== nothing
                push!(x, ArchGDAL.getx(point, 0))
                push!(y, ArchGDAL.gety(point, 0))
            end
        end
        
        # Plot only if we have valid coordinates
        if length(x) >= 3
            plot!(p, x, y,
                  seriestype=:shape,
                  fillalpha=fill_alpha,
                  fillcolor=fill_color,
                  linecolor=line_color,
                  linewidth=line_width,
                  label="")
        end
        
    catch e
        println("Error processing polygon: $e")
    end
end

In [ ]:
# Usage examples
function main()
    # Set file path (change to your actual shp file path)
    file_path = "path/to/your/korea_map.shp"
    
    # Basic map plotting
    println("Plotting Korea map...")
    map_plot = plot_korea_map(file_path)
    
    if map_plot !== nothing
        # Save the map
        savefig(map_plot, "korea_map.png")
        println("Map saved as 'korea_map.png'")
        
        # Display the map
        display(map_plot)
    end
    
    # Create maps with different styles
    println("\nCreating maps with different styles...")
    
    # Style 1: Dark colors
    map1 = plot_korea_map(file_path,
                         map_title="Korea - Dark Style",
                         fill_color=:darkgreen,
                         line_color=:black,
                         line_width=1.5,
                         fill_alpha=0.8)
    
    # Style 2: Light colors
    map2 = plot_korea_map(file_path,
                         map_title="Korea - Light Style",
                         fill_color=:lightcoral,
                         line_color=:darkred,
                         line_width=1.0,
                         fill_alpha=0.6)
    
    # Style 3: Blue theme
    map3 = plot_korea_map(file_path,
                         map_title="Korea - Blue Theme",
                         fill_color=:skyblue,
                         line_color=:navy,
                         line_width=1.2,
                         fill_alpha=0.7)
    
    if map1 !== nothing && map2 !== nothing && map3 !== nothing
        # Combine multiple maps in one plot
        combined_plot = plot(map1, map2, map3, layout=(1,3), size=(3600, 900))
        savefig(combined_plot, "korea_maps_comparison.png")
        display(combined_plot)
    end
end

In [ ]:
# Advanced function for customizing map appearance
function plot_korea_map_advanced(file_path::String;
                               map_title::String="Korea Map",
                               fill_colors=[:lightblue],
                               line_color=:navy,
                               line_width=1.2,
                               fill_alpha=0.7,
                               image_size=(1200, 900),
                               show_grid=true,
                               background_color=:white)
    
    p = plot(size=image_size, 
             background_color=background_color,
             margin=5Plots.mm)
    
    try
        ArchGDAL.read(file_path) do dataset
            polygon_count = 0
            color_index = 1
            
            for i in 0:(ArchGDAL.nlayer(dataset)-1)
                layer = ArchGDAL.getlayer(dataset, i)
                
                for feature in layer
                    geom = ArchGDAL.getgeom(feature, 0)
                    
                    if geom === nothing
                        continue
                    end
                    
                    # Cycle through colors if multiple colors provided
                    current_color = fill_colors[((color_index - 1) % length(fill_colors)) + 1]
                    
                    geom_type = ArchGDAL.getgeomtype(geom)
                    
                    if geom_type == ArchGDAL.wkbPolygon
                        plot_polygon!(p, geom, current_color, line_color, line_width, fill_alpha)
                        polygon_count += 1
                        color_index += 1
                        
                    elseif geom_type == ArchGDAL.wkbMultiPolygon
                        n_polygons = ArchGDAL.ngeom(geom)
                        for k in 0:(n_polygons-1)
                            poly = ArchGDAL.getgeom(geom, k)
                            plot_polygon!(p, poly, current_color, line_color, line_width, fill_alpha)
                            polygon_count += 1
                        end
                        color_index += 1
                    end
                end
            end
        end
        
    catch e
        println("Error: $e")
        return nothing
    end
    
    plot!(title=map_title,
          titlefontsize=16,
          xlabel="Longitude",
          ylabel="Latitude",
          xlabelfontsize=12,
          ylabelfontsize=12,
          aspect_ratio=:equal,
          grid=show_grid,
          gridwidth=1,
          gridcolor=:lightgray,
          gridalpha=0.3,
          legend=false,
          dpi=300)
    
    return p
end

# Execute main function
# main()

In [ ]:
# Basic usage
map_plot = plot_korea_map(file_path)

# Custom styling
custom_map = plot_korea_map(file_path,
                           map_title="South Korea",
                           fill_color=:lightgreen,
                           line_color=:darkgreen,
                           line_width=2.0,
                           fill_alpha=0.7)

# Advanced multi-color map
colorful_map = plot_korea_map_advanced(file_path,
                                     fill_colors=[:lightblue, :lightgreen, :lightcoral],
                                     map_title="Korea - Multi-Color")

## 지도 그리기 3
엑셀 값 병합

In [ ]:
using ArchGDAL
using Plots
using XLSX
using Colors
using ColorSchemes

In [ ]:
# 사용 가능한 컬러스킴 확인 함수
function check_available_colorschemes()
    println("Available ColorSchemes:")
    # colorschemes 딕셔너리에서 직접 키들을 가져옴
    try
        for scheme_name in keys(ColorSchemes.colorschemes)
            println("  - ColorSchemes.$scheme_name")
        end
    catch e
        println("Error accessing colorschemes: $e")
        println("Try these common ones:")
        common_schemes = ["viridis", "plasma", "inferno", "magma", "blues", "reds", "greens", 
                         "coolwarm", "seaborn_icefire", "RdYlBu", "RdBu", "BrBG"]
        for scheme in common_schemes
            println("  - ColorSchemes.$scheme")
        end
    end
end

In [ ]:
# 특정 컬러스킴이 존재하는지 확인하는 함수
function test_colorscheme(scheme_name::String)
    try
        scheme = getfield(ColorSchemes, Symbol(scheme_name))
        println("✓ ColorSchemes.$scheme_name is available")
        return true
    catch e
        println("✗ ColorSchemes.$scheme_name is NOT available")
        return false
    end
end

In [ ]:
test_colorscheme("RdYlBu")
test_colorscheme("viridis")
test_colorscheme("coolwarm")

In [ ]:
# Function to read Excel data and create value mapping
function read_excel_data(excel_path::String)
    println("Reading Excel file: $excel_path")
    
    # Use direct sheet access method
    xf = XLSX.readxlsx(excel_path)
    sheet = xf["Sheet1"]
    
    region_values = Dict{String, Float64}()
    
    # Try to read data manually
    row_num = 2  # Start from row 2 (skip header)
    while true
        try
            region_cell = sheet[row_num, 1]  # Column A
            value_cell = sheet[row_num, 2]   # Column B
            
            if region_cell === nothing && value_cell === nothing
                break  # End of data
            end
            
            if region_cell !== nothing && value_cell !== nothing
                region_name = strip(string(region_cell))
                value = float(value_cell)
                region_values[region_name] = value
            end
            
            row_num += 1
            
        catch e
            # If we can't read the cell, we've reached the end
            break
        end
    end
    
    println("Successfully loaded $(length(region_values)) regions from Excel file")
    return region_values
end

In [ ]:
# Excel 데이터만 먼저 테스트
excel_path = "/Users/mkim/Library/Mobile Documents/com~apple~CloudDocs/2025 연구/CA2025/Korea_map/Korea_map.xlsx"
region_data = read_excel_data(excel_path)

# 결과 확인
println("Total regions loaded: ", length(region_data))

# 처음 5개 샘플 출력
count = 0
for (region, value) in region_data
    println("$region: $value")
    count += 1
    if count >= 5
        break
    end
end

In [ ]:
# Function to normalize values and get color
function get_color_for_value(value::Float64, min_val::Float64, max_val::Float64, 
    colorscheme=ColorSchemes.RdYlBu)  # RdYlBu_r → RdYlBu
    
    # Normalize value to 0-1 range
    normalized = (value - min_val) / (max_val - min_val)

    # RdYlBu는 파란색(낮은값)에서 빨간색(높은값)으로 가므로 역순으로 만들기
    normalized = 1.0 - normalized  # 역순으로 변경

    # Clamp to ensure it's between 0 and 1
    normalized = clamp(normalized, 0.0, 1.0)

    # Get color from colorscheme
    return get(colorscheme, normalized)
end

In [ ]:
# Enhanced function to plot Korea map with Excel data coloring
function plot_korea_choropleth(shp_path::String, excel_path::String;
   map_title::String="Korea Choropleth Map",
   colorscheme=ColorSchemes.RdYlBu,  # 수정된 컬러스킴
   line_color=:white,
   line_width=0.5,
   fill_alpha=0.8,
   image_size=(1400, 1000),
   default_color=:lightgray,
   show_colorbar=false)

   # Read Excel data
   println("Reading Excel data...")
   region_values = read_excel_data(excel_path)

   if isempty(region_values)
      println("No data found in Excel file!")
      return nothing
   end

   # Get min and max values for color scaling
   region_values_list = collect(values(region_values))
   min_val = minimum(region_values_list)
   max_val = maximum(region_values_list)

   println("Data range: $(min_val) to $(max_val)")
   println("Number of regions with data: $(length(region_values))")

   # Initialize plot
   p = plot(size=image_size, background_color=:white, margin=5Plots.mm)

   matched_regions = 0
   total_polygons = 0

   try
      ArchGDAL.read(shp_path) do dataset
         for i in 0:(ArchGDAL.nlayer(dataset)-1)
            layer = ArchGDAL.getlayer(dataset, i)

            for feature in layer
               geom = ArchGDAL.getgeom(feature, 0)

               if geom === nothing
                  continue
               end

               total_polygons += 1

               # Try to get region name from different possible fields
               region_name = nothing
               for field_name in ["SIG_ENG_NM", "NAME_ENG", "ENG_NM", "ENGNAME", "NAME"]
                  try
                     field_value = ArchGDAL.getfield(feature, field_name)
                     if field_value !== nothing
                        region_name = string(field_value)
                        break
                     end
                  catch
                     continue
                  end
               end

               # Determine fill color based on data
               fill_color = default_color

               if region_name !== nothing && haskey(region_values, region_name)
                  value = region_values[region_name]
                  fill_color = get_color_for_value(value, min_val, max_val, colorscheme)
                  matched_regions += 1
               end

               # Plot the geometry
               geom_type = ArchGDAL.getgeomtype(geom)

               if geom_type == ArchGDAL.wkbPolygon
                  plot_polygon!(p, geom, fill_color, line_color, line_width, fill_alpha)

               elseif geom_type == ArchGDAL.wkbMultiPolygon
                  n_polygons = ArchGDAL.ngeom(geom)
                  for k in 0:(n_polygons-1)
                     poly = ArchGDAL.getgeom(geom, k)
                     plot_polygon!(p, poly, fill_color, line_color, line_width, fill_alpha)
                  end
               end
            end
         end
      end

      println("Matched $(matched_regions) out of $(total_polygons) regions")

   catch e
      println("Error processing shapefile: $e")
      return nothing
   end

   # Style the main plot
   plot!(title=map_title, titlefontsize=16,
   xlabel="Longitude", ylabel="Latitude",
   xlabelfontsize=12, ylabelfontsize=12,
   aspect_ratio=:equal, grid=false, legend=false,
   dpi=300)

   # Add colorbar if requested
   if show_colorbar
      # Create a separate colorbar plot
      colorbar_values = range(min_val, max_val, length=100)
      colorbar_colors = [get_color_for_value(v, min_val, max_val, colorscheme) for v in colorbar_values]

      colorbar_plot = plot(collect(colorbar_values), ones(length(colorbar_values)),
      seriestype=:scatter,
      markercolor=colorbar_colors, markersize=3, markerstrokewidth=0,
      xlabel="Value", ylabel="",
      title="Color Scale", titlefontsize=12,
      xlabelfontsize=10,
      size=(300, 100), grid=false, legend=false,
      margin=2Plots.mm)

      # Combine main plot and colorbar
      combined_plot = plot(p, colorbar_plot, layout=@layout([a{0.8w} b{0.2w}]),
      size=(image_size[1] + 300, image_size[2]))

      return combined_plot
   end

   return p
end


In [ ]:
# Helper function to plot individual polygons
function plot_polygon!(p, geom, fill_color, line_color, line_width, fill_alpha)
    try
        ring = ArchGDAL.getgeom(geom, 0)
        if ring === nothing
            return
        end
        
        n_points = ArchGDAL.ngeom(ring)
        if n_points < 3
            return
        end
        
        x = Float64[]
        y = Float64[]
        
        for j in 0:(n_points-1)
            point = ArchGDAL.getgeom(ring, j)
            if point !== nothing
                push!(x, ArchGDAL.getx(point, 0))
                push!(y, ArchGDAL.gety(point, 0))
            end
        end
        
        if length(x) >= 3
            plot!(p, x, y,
                  seriestype=:shape,
                  fillalpha=fill_alpha,
                  fillcolor=fill_color,
                  linecolor=line_color,
                  linewidth=line_width,
                  label="")
        end
        
    catch e
        println("Error processing polygon: $e")
    end
end

In [ ]:
# Function to create different color schemes
function plot_multiple_color_schemes(shp_path::String, excel_path::String)
    # 사용 가능한 컬러스킴들로 변경
    schemes = [
        (ColorSchemes.viridis, "Viridis (Purple to Yellow)"),
        (ColorSchemes.plasma, "Plasma (Purple to Pink)"),
        (ColorSchemes.RdYlBu, "Red-Yellow-Blue (Red=High)"),
        (ColorSchemes.blues, "Blues (Light to Dark Blue)")
    ]
    
    plots_array = []
    
    for (scheme, title) in schemes
        p = plot_korea_choropleth(shp_path, excel_path,
                                map_title=title,
                                colorscheme=scheme,
                                image_size=(700, 500),
                                show_colorbar=false)
        if p !== nothing
            push!(plots_array, p)
        end
    end
    
    if length(plots_array) >= 4
        combined = plot(plots_array[1], plots_array[2], plots_array[3], plots_array[4],
                       layout=(2,2), size=(1400, 1000))
        return combined
    else
        return nothing
    end
end

In [ ]:
# Main execution function
function main()
    # File paths (adjust these to your actual file paths)
    shp_path = "/Users/mkim/Library/Mobile Documents/com~apple~CloudDocs/2025 연구/CA2025/Korea_map/Korea_map.shp"
    excel_path = "/Users/mkim/Library/Mobile Documents/com~apple~CloudDocs/2025 연구/CA2025/Korea_map/Korea_map.xlsx"
    
    println("Creating choropleth map...")
    
    # Create main choropleth map
    choropleth_map = plot_korea_choropleth(shp_path, excel_path,
                                         map_title="Korea - CA Model Results",
                                         colorscheme=ColorSchemes.RdYlBu,
                                         show_colorbar=false)
    
    if choropleth_map !== nothing
        # Save the map
        savefig(choropleth_map, "korea_choropleth_map.png")
        println("Choropleth map saved as 'korea_choropleth_map.png'")
        
        # Display the map
        display(choropleth_map)
        
        # Create comparison with different color schemes
        println("Creating comparison with different color schemes...")
        comparison_map = plot_multiple_color_schemes(shp_path, excel_path)
        
        if comparison_map !== nothing
            savefig(comparison_map, "korea_choropleth_comparison.png")
            println("Comparison map saved as 'korea_choropleth_comparison.png'")
            display(comparison_map)
        end
    else
        println("Failed to create choropleth map. Check your file paths and data.")
    end
end

In [ ]:
# Custom color scheme example
function create_blue_red_map(shp_path::String, excel_path::String)
    # Create a custom blue-to-red color scheme
    custom_scheme = ColorScheme([
        RGB(0.0, 0.0, 1.0),    # Pure blue (low values)
        RGB(0.5, 0.5, 1.0),    # Light blue
        RGB(1.0, 1.0, 1.0),    # White (middle values)  
        RGB(1.0, 0.5, 0.5),    # Light red
        RGB(1.0, 0.0, 0.0)     # Pure red (high values)
    ])
    
    map_plot = plot_korea_choropleth(shp_path, excel_path,
                                   map_title="Korea - Blue (Low) to Red (High)",
                                   colorscheme=custom_scheme,
                                   show_colorbar=false)
    
    return map_plot
end

In [ ]:
# 파일 경로 설정 (shapefile 경로만 실제 경로로 수정하세요)
shp_path = "/Users/mkim/Library/Mobile Documents/com~apple~CloudDocs/2025 연구/CA2025/Korea_map/Korea_map.shp"
excel_path = "/Users/mkim/Library/Mobile Documents/com~apple~CloudDocs/2025 연구/CA2025/Korea_map/Korea_map.xlsx"

# 지도 생성
choropleth_map = plot_korea_choropleth(shp_path, excel_path,
                                     map_title="Korea - CA Model Results",
                                     colorscheme=ColorSchemes.RdYlBu,
                                     show_colorbar=false)

# 결과 표시
display(choropleth_map)

## 지도 그리기
단체로 그리기

In [ ]:
using ArchGDAL
using Plots
using CSV
using DataFrames
using Colors
using ColorSchemes

In [ ]:
# Function to read CSV data and create value mapping for a specific column
function read_csv_column_data(csv_path::String, column_name::String)
    println("Reading CSV file: $csv_path for column: $column_name")
    
    # Read CSV file
    df = CSV.read(csv_path, DataFrame)
    
    region_values = Dict{String, Float64}()
    
    # Check if the column exists
    if !(column_name in names(df))
        println("Error: Column '$column_name' not found in CSV file")
        return region_values
    end
    
    # Create dictionary mapping region names to values
    for row in eachrow(df)
        region_name = strip(string(row[1]))  # First column is region name
        value = row[column_name]
        
        if !ismissing(value) && !isnan(value)
            region_values[region_name] = Float64(value)
        end
    end
    
    println("Successfully loaded $(length(region_values)) regions for column: $column_name")
    return region_values
end

In [ ]:
# Function to normalize values and get color
function get_color_for_value(value::Float64, min_val::Float64, max_val::Float64, 
                           #colorscheme=ColorSchemes.RdYlBu
                           colorscheme=ColorSchemes.Spectral_11)
    # Normalize value to 0-1 range
    normalized = (value - min_val) / (max_val - min_val)
    # RdYlBu는 파란색(낮은값)에서 빨간색(높은값)으로 가므로 역순으로 만들기
    normalized = 1.0 - normalized  # 역순으로 변경
    # Clamp to ensure it's between 0 and 1
    normalized = clamp(normalized, 0.0, 1.0)
    # Get color from colorscheme
    return get(colorscheme, normalized)
end


In [ ]:
# # 생생한 다채로운 색상 스킴 생성 (이미지와 유사하게)
# function create_vibrant_colorscheme()
#     # 채도를 낮춘 부드러운 색상들 정의
#     vibrant_colors = [
#         RGB(0.2, 0.4, 0.7),     # 부드러운 파란색 (채도 낮춤)
#         RGB(0.3, 0.6, 0.8),     # 부드러운 하늘색
#         RGB(0.4, 0.7, 0.6),     # 부드러운 청록색
#         RGB(0.5, 0.8, 0.5),     # 부드러운 연두색
#         RGB(0.7, 0.8, 0.4),     # 부드러운 노란색
#         RGB(0.8, 0.7, 0.3),     # 부드러운 주황-노랑
#         RGB(0.8, 0.6, 0.3),     # 부드러운 주황색
#         RGB(0.8, 0.4, 0.2),     # 부드러운 빨간색
#         RGB(0.7, 0.3, 0.3)      # 부드러운 진한 빨간색
#     ]
    
#     return ColorScheme(vibrant_colors)
# end


# # Function to normalize values and get color (수정됨: 새로운 색상 스킴 적용)
# function get_color_for_value(value::Float64, min_val::Float64, max_val::Float64, 
#                            colorscheme=create_vibrant_colorscheme())  # 수정됨: 기본값을 생생한 색상으로 변경
#     # Normalize value to 0-1 range
#     normalized = (value - min_val) / (max_val - min_val)
    
#     # Clamp to ensure it's between 0 and 1
#     normalized = clamp(normalized, 0.0, 1.0)
    
#     # Get color from colorscheme (역순 없이 직접 사용)
#     return get(colorscheme, normalized)
# end

In [ ]:
# Enhanced function to plot Korea map with CSV column data coloring
function plot_korea_choropleth_csv(shp_path::String, csv_path::String, column_name::String;
                                 map_title::String="Korea Choropleth Map",
                                 #colorscheme=ColorSchemes.RdYlBu,
                                 #colorscheme=create_vibrant_colorscheme(),
                                 colorscheme=ColorSchemes.Spectral_11,
                                 line_color=:white,
                                 line_width=0.5,
                                 fill_alpha=0.8,
                                 image_size=(1400, 1000),
                                 default_color=:lightgray,
                                 show_colorbar=false)
    
    # Read CSV data for specific column
    println("Reading CSV column data...")
    region_values = read_csv_column_data(csv_path, column_name)
    
    if isempty(region_values)
        println("No data found for column: $column_name")
        return nothing
    end
    
    # Get min and max values for color scaling
    region_values_list = collect(values(region_values))
    min_val = minimum(region_values_list)
    max_val = maximum(region_values_list)
    
    println("Data range for $column_name: $(min_val) to $(max_val)")
    println("Number of regions with data: $(length(region_values))")
    
    # Initialize plot
    # 투명 배경으로 플롯 초기화 (수정됨: :white → :transparent)
    p = plot(size=image_size, 
             background_color=:transparent,  # 투명 배경
             margin=5Plots.mm)
    
    matched_regions = 0
    total_polygons = 0
    
    try
        ArchGDAL.read(shp_path) do dataset
            for i in 0:(ArchGDAL.nlayer(dataset)-1)
                layer = ArchGDAL.getlayer(dataset, i)
                
                for feature in layer
                    geom = ArchGDAL.getgeom(feature, 0)
                    
                    if geom === nothing
                        continue
                    end
                    
                    total_polygons += 1
                    
                    # Try to get region name from different possible fields
                    region_name = nothing
                    for field_name in ["SIG_ENG_NM", "NAME_ENG", "ENG_NM", "ENGNAME", "NAME"]
                        try
                            field_value = ArchGDAL.getfield(feature, field_name)
                            if field_value !== nothing
                                region_name = string(field_value)
                                break
                            end
                        catch
                            continue
                        end
                    end
                    
                    # Determine fill color based on data
                    fill_color = default_color
                    
                    if region_name !== nothing && haskey(region_values, region_name)
                        value = region_values[region_name]
                        fill_color = get_color_for_value(value, min_val, max_val, colorscheme)
                        matched_regions += 1
                    end
                    
                    # Plot the geometry
                    geom_type = ArchGDAL.getgeomtype(geom)
                    
                    if geom_type == ArchGDAL.wkbPolygon
                        plot_polygon!(p, geom, fill_color, line_color, line_width, fill_alpha)
                        
                    elseif geom_type == ArchGDAL.wkbMultiPolygon
                        n_polygons = ArchGDAL.ngeom(geom)
                        for k in 0:(n_polygons-1)
                            poly = ArchGDAL.getgeom(geom, k)
                            plot_polygon!(p, poly, fill_color, line_color, line_width, fill_alpha)
                        end
                    end
                end
            end
        end
        
        println("Matched $(matched_regions) out of $(total_polygons) regions for column: $column_name")
        
    catch e
        println("Error processing shapefile: $e")
        return nothing
    end
    
    # Style the main plot
    plot!(title=map_title,
          titlefontsize=16,
          xlabel="Longitude",
          ylabel="Latitude",
          xlabelfontsize=12,
          ylabelfontsize=12,
          aspect_ratio=:equal,
          grid=false,
          legend=false,
          dpi=300)
    
    # Add colorbar if requested
    if show_colorbar
        # Create a separate colorbar plot
        colorbar_values = range(min_val, max_val, length=100)
        colorbar_colors = [get_color_for_value(v, min_val, max_val, colorscheme) for v in colorbar_values]
        
        colorbar_plot = plot(collect(colorbar_values), ones(length(colorbar_values)),
                           seriestype=:scatter,
                           markercolor=colorbar_colors,
                           markersize=3,
                           markerstrokewidth=0,
                           xlabel="Value",
                           ylabel="",
                           title="Color Scale",
                           titlefontsize=12,
                           xlabelfontsize=10,
                           size=(300, 100),
                           grid=false,
                           legend=false,
                           margin=2Plots.mm)
        
        # Combine main plot and colorbar
        combined_plot = plot(p, colorbar_plot, 
                           layout=@layout([a{0.8w} b{0.2w}]),
                           size=(image_size[1] + 300, image_size[2]))
        
        return combined_plot
    end
    
    return p
end

In [ ]:
# Helper function to plot individual polygons
function plot_polygon!(p, geom, fill_color, line_color, line_width, fill_alpha)
    try
        ring = ArchGDAL.getgeom(geom, 0)
        if ring === nothing
            return
        end
        
        n_points = ArchGDAL.ngeom(ring)
        if n_points < 3
            return
        end
        
        x = Float64[]
        y = Float64[]
        
        for j in 0:(n_points-1)
            point = ArchGDAL.getgeom(ring, j)
            if point !== nothing
                push!(x, ArchGDAL.getx(point, 0))
                push!(y, ArchGDAL.gety(point, 0))
            end
        end
        
        if length(x) >= 3
            plot!(p, x, y,
                  seriestype=:shape,
                  fillalpha=fill_alpha,
                  fillcolor=fill_color,
                  linecolor=line_color,
                  linewidth=line_width,
                  label="")
        end
        
    catch e
        println("Error processing polygon: $e")
    end
end

In [ ]:
# Function to create maps for all columns in CSV
function create_all_column_maps(shp_path::String, csv_path::String; 
                              output_dir::String="./maps/",
                              #colorscheme=ColorSchemes.RdYlBu,
                              #colorscheme=create_vibrant_colorscheme()
                              colorscheme=ColorSchemes.Spectral_11)
    
    # Create output directory if it doesn't exist
    if !isdir(output_dir)
        mkdir(output_dir)
    end
    
    # Read CSV to get column names
    df = CSV.read(csv_path, DataFrame)
    column_names = names(df)[2:end]  # Skip first column (region names)
    
    println("Found $(length(column_names)) columns to process")
    
    successful_maps = 0
    failed_maps = 0
    
    for (index, column_name) in enumerate(column_names)
        println("\n--- Processing column $(index)/$(length(column_names)): $column_name ---")
        
        try
            # Create map for this column
            map_plot = plot_korea_choropleth_csv(shp_path, csv_path, column_name,
                                               map_title="$column_name",
                                               colorscheme=colorscheme,
                                               show_colorbar=false)
            
            if map_plot !== nothing
                # Save with column name as filename
                output_filename = joinpath(output_dir, "$(column_name).png")
                savefig(map_plot, output_filename)
                println("✓ Saved: $output_filename")
                successful_maps += 1
            else
                println("✗ Failed to create map for column: $column_name")
                failed_maps += 1
            end
            
        catch e
            println("✗ Error processing column $column_name: $e")
            failed_maps += 1
        end
    end
    
    println("\n=== Summary ===")
    println("Successfully created: $successful_maps maps")
    println("Failed: $failed_maps maps")
    println("Total processed: $(successful_maps + failed_maps) columns")
    
    return successful_maps, failed_maps
end

In [ ]:
# Function to create a sample of maps (first N columns)
function create_sample_maps(shp_path::String, csv_path::String, n_samples::Int=5;
                          output_dir::String="./sample_maps/",
                          #colorscheme=ColorSchemes.RdYlBu
                          #colorscheme=create_vibrant_colorscheme()
                          colorscheme=ColorSchemes.Spectral_11)
    
    # Create output directory if it doesn't exist
    if !isdir(output_dir)
        mkdir(output_dir)
    end
    
    # Read CSV to get column names
    df = CSV.read(csv_path, DataFrame)
    column_names = names(df)[2:end]  # Skip first column (region names)
    
    # Take first n_samples columns
    sample_columns = column_names[1:min(n_samples, length(column_names))]
    
    println("Creating sample maps for $(length(sample_columns)) columns")
    
    for (index, column_name) in enumerate(sample_columns)
        println("\n--- Processing sample $(index)/$(length(sample_columns)): $column_name ---")
        
        try
            map_plot = plot_korea_choropleth_csv(shp_path, csv_path, column_name,
                                               map_title="$column_name",
                                               colorscheme=colorscheme,
                                               show_colorbar=false)
            
            if map_plot !== nothing
                output_filename = joinpath(output_dir, "$(column_name).png")
                savefig(map_plot, output_filename)
                println("✓ Saved sample: $output_filename")
                
                # Also display the first map
                if index == 1
                    display(map_plot)
                end
            end
            
        catch e
            println("✗ Error creating sample map for $column_name: $e")
        end
    end
end

In [ ]:
# # Main execution function for multiple maps
# function main_multi_maps()
#     # File paths (adjust these to your actual file paths)
#     shp_path = "path/to/your/korea_map.shp"  # 실제 shapefile 경로
#     csv_path = "Maxent_76_latin_S.csv"       # CSV 파일 경로
    
#     println("=== Creating Multiple Korea Choropleth Maps ===")
    
#     # Option 1: Create sample maps first (first 5 columns)
#     println("\n1. Creating sample maps (first 5 columns)...")
#     create_sample_maps(shp_path, csv_path, 5)
    
#     # Option 2: Create all maps (uncomment to use)
#     # println("\n2. Creating all column maps...")
#     # create_all_column_maps(shp_path, csv_path)
# end

In [ ]:
# Execute sample creation
# main_multi_maps()

shp_path = "/Users/mkim/Library/Mobile Documents/com~apple~CloudDocs/2025 연구/CA2025/Korea_map/Korea_map.shp"
csv_path = "/Users/mkim/Library/Mobile Documents/com~apple~CloudDocs/2025 연구/CA2025/Data_20250414/TES/Maxent_76_latin_S.csv"

# 옵션 1: 샘플 지도 생성 (처음 5개 열만 - 테스트용)
# create_sample_maps(shp_path, csv_path, 5)

# # 옵션 2: 모든 열에 대해 지도 생성 (81개 파일)
create_all_column_maps(shp_path, csv_path)

# # 옵션 3: 특정 열만 지도 생성
# column_name = "126_126_126_126"
# map_plot = plot_korea_choropleth_csv(shp_path, csv_path, column_name,
#                                    map_title="Korea - $column_name")
# savefig(map_plot, "$(column_name).png")